<a href="https://colab.research.google.com/github/Prinzenrolle10/Credit-Card-Fraud-Detection/blob/main/Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fraud-Detection mit Machine Learning und Deep Learning**


**Projektziel und Problemstellung**

---

In diesem Notebook wird ein Machine Learning Modell zur Erkennung von Kreditkartenbetrug entwickelt. Die grösste Herausforderung dieses Datensatzes ist die extreme Klassen-Unbalance. Betrugsfälle machen nur ca. 0.17% der Daten aus.
Es werden zwei Ansätze verglichen: ein klassisches, baumbasiertes Modell (XGBoost) und ein Deep Learning Ansatz zur Anomalieerkennung (Autoencoder).

In [ ]:
import pandas as pd
import numpy as np
import kagglehub
from sklearn.preprocessing import StandardScaler

In [ ]:
# Daten laden
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(f"{path}/creditcard.csv")

print(f"Shape: {df.shape}")
df.head()

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
# Feature Engineering
df['event_timestamp'] = pd.to_datetime(df['Time'], unit='s', utc=True)
df = df.sort_values('event_timestamp').reset_index(drop=True)

df['time_diff'] = df['Time'].diff().fillna(0)
df['hour'] = (df['Time'] // 3600) % 24
df['amount_log'] = np.log1p(df['Amount'])

# Spalten umbenennen
v_cols = {f'V{i}': f'v{i}' for i in range(1, 29)}
df = df.rename(columns=v_cols)
df = df.rename(columns={'Amount': 'amount', 'Class': 'class'})

# Rollierende Features (1 Stunde Fenster)
df = df.set_index('event_timestamp', drop=False)
df['trans_count_1h'] = df['amount'].rolling('3600s').count().fillna(0.0)
df['amount_trans_count_1h'] = df['amount'].rolling('3600s').sum().fillna(0.0)
df = df.reset_index(drop=True)

# Feature-Liste definieren
feature_cols = [
    'time_diff', 'hour', 'trans_count_1h', 'amount_log', 'amount', 'amount_trans_count_1h'
] + [f'v{i}' for i in range(1, 29)]

print(f"Shape: {df.shape}")
df.head()

Shape: (284807, 37)


,Time,v1,v2,v3,v4,v5,v6,v7,v8,v9,...,v27,v28,amount,class,event_timestamp,time_diff,hour,amount_log,trans_count_1h,amount_trans_count_1h
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.133558,-0.021053,149.62,0,1970-01-01 00:00:00+00:00,0.0,0.0,5.014760,1.0,149.62
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.008983,0.014724,2.69,0,1970-01-01 00:00:00+00:00,0.0,0.0,1.305626,2.0,152.31
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.055353,-0.059752,378.66,0,1970-01-01 00:00:01+00:00,1.0,0.0,5.939276,3.0,530.97
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.062723,0.061458,123.50,0,1970-01-01 00:00:01+00:00,0.0,0.0,4.824306,4.0,654.47
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.219422,0.215153,69.99,0,1970-01-01 00:00:02+00:00,1.0,0.0,4.262539,5.0,724.46


In [ ]:
# Train-Test-Split (chronologisch - 60-20-20)
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()


In [ ]:
# Skalierung
scaler = StandardScaler()

X_train = scaler.fit_transform(train_df[feature_cols])
y_train = train_df['class'].values

X_val = scaler.transform(val_df[feature_cols])
y_val = val_df['class'].values

X_test = scaler.transform(test_df[feature_cols])
y_test = test_df['class'].values

print(f"Train Shape: {X_train.shape}, Fraud Fälle: {sum(y_train)}")
print(f"Val Shape:   {X_val.shape}, Fraud Fälle: {sum(y_val)}")
print(f"Test Shape:  {X_test.shape}, Fraud Fälle: {sum(y_test)}")

Train Shape: (170884, 34), Fraud Fälle: 360
Val Shape:   (56961, 34), Fraud Fälle: 57
Test Shape:  (56962, 34), Fraud Fälle: 75


**ML-Pfad mit XGBoost und Optuna**

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 17.0 MB/s eta 0:00:00


In [ ]:
import logging
import sys
import numpy as np
import xgboost as xgb
import optuna
from sklearn.metrics import average_precision_score
from typing import Tuple, Dict, Any

In [ ]:
# Logging Setup
logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    force = True,
    stream = sys.stdout
)

logger = logging.getLogger(__name__)

Warum AUCPR statt ROC-AUC?

---

Da die Daten hochgradig unbalanciert sind, würde die klassische ROC-AUC-Metrik durch die Vielzahl der normalen Transaktionen ein trügerisch gutes Ergebnis liefern. Optuna wird daher auf den AUCPR (Area Under the Precision-Recall Curve) optimiert, da diese Metrik stark auf die Minderheisklasse (Betrufsfälle) fokussiert ist.

In [ ]:
# Optuna Objective Function
def objective(
    trial: optuna.Trial,
    X_tain: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
) -> float:

  """
  Optuna Objective Funktion für das Tuning von XGBoost.
  Sucht nach den besten Hyperparametern für imbalanced Fraud-Daten.
  """

  # dynamische Berechnung der Balance für 'scale_pos_weight' als Anhaltspunkt
  # ratio = Anzahl normale Transaktionen / Anzahl Fraud
  ratio = float(np.sum(y_train == 0)) / float(np.sum(y_train == 1))

  # Suchraum für die Hyperparameter
  params = {
      "objective": "binary:logistic",
      "eval_metric": "aucpr",
      "tree_method": "hist",
      "device": "cuda",                # GPU in Colab
      "random_state": 42,
      "max_depth": trial.suggest_int("max_depth", 3, 10),
      "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.03, log=True),
      "n_estimators": trial.suggest_int("n_estimators", 100, 500),
      "subsample": trial.suggest_float("subsample", 0.5, 1.0),
      "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
      "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, ratio * 1.5),
  }

  # Modell intialisieren und trainieren
  model = xgb.XGBClassifier(**params)

  model.fit(
      X_train,
      y_train,
      eval_set=[(X_val, y_val)],
      verbose = False
  )

  # Vorhersagen auf dem Validierungsset
  pred_proba = model.predict_proba(X_val)[:, 1]

  # AUCPR-Metrik
  aucpr = average_precision_score(y_val, pred_proba)

  return aucpr

In [ ]:
# Main Tuning Function
def tune_and_train_xgboost(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    n_trials: int = 20
) -> Tuple[xgb.XGBClassifier, Dict[str, Any]]:
    """
    Führt das Hyperparameter-Tuning durch und gibt das beste Modell zurück.
    """
    logger.info(f"Starte Tuning mit {n_trials} Trials...")

    # Optuna Study erstellen
    study = optuna.create_study(direction="maximize", study_name="XGBoost_Fraud_Detection")

    # Lambda-Funktion gibt Parameter and die Objective-Funktion weiter
    study.optimize(
        lambda trial: objective(trial, X_train, y_train, X_val, y_val),
        n_trials=n_trials
    )

    logger.info("Tuning abgeschlossen.")
    logger.info(f"Bester AUCPR auf Validationset: {study.best_value:.4f}")
    logger.info(f"Beste Hyperparameter: {study.best_params}")

    # Finales Training mit den besten Hyperparametern
    best_params = study.best_params
    best_params["objective"] = "binary:logistic"
    best_params["eval_metric"] = "aucpr"
    best_params["tree_method"] = "hist"
    best_params["device"] = "cuda"
    best_params["random_state"] = 42

    logger.info("Training des finalen Modells mit den besten Hyperparametern.")

    best_xgb_model = xgb.XGBClassifier(**best_params)
    best_xgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    return best_xgb_model, best_params

In [ ]:
# Tuning durchführen (30 Runden als Start)
best_model, best_parameters = tune_and_train_xgboost(X_train, y_train, X_val, y_val, n_trials=30)

2026-03-30 17:40:07,638 - INFO - Starte Tuning mit 30 Trials...


[I 2026-03-30 17:40:07,639] A new study created in memory with name: XGBoost_Fraud_Detection
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [17:40:10] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-03-30 17:40:10,288] Trial 0 finished with value: 0.7386421232067858 and parameters: {'max_depth': 10, 'learning_rate': 0.015343506465359261, 'n_estimators': 100, 'subsample': 0.8409236537874476, 'colsample_bytree': 0.799067559198293, 'scale_pos_weight': 634.2372381103538}. Best is trial 0 with value: 0.7386421232067858.
[I 2026-03-30 17:40:12

2026-03-30 17:41:24,517 - INFO - Tuning abgeschlossen.
2026-03-30 17:41:24,518 - INFO - Bester AUCPR auf Validationset: 0.7879
2026-03-30 17:41:24,519 - INFO - Beste Hyperparameter: {'max_depth': 10, 'learning_rate': 0.02655600975024074, 'n_estimators': 352, 'subsample': 0.559679617005939, 'colsample_bytree': 0.7580740651136493, 'scale_pos_weight': 51.49509015224637}
2026-03-30 17:41:24,519 - INFO - Training des finalen Modells mit den besten Hyperparametern.


**Deep Learning Ansatz**

---

Als Vergleich zum XGBoost wird einen Autoencoder gebaut. Die Idee ist, dass der Autoencoder ausschliesslich auf normale Transaktionen trainiert wird. Dadurch lernt er das Grundrauschen kennen. Bei einer Betrugsaktion sollte dann das Netz scheitern, diese zu rekonstruieren, was zu einem hohen Mean Squared Error (MSE) führt.

In [ ]:
import logging
import sys
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from sklearn.metrics import average_precision_score
from typing import Tuple

In [ ]:
# Logging Setup
logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    force = True,
    stream = sys.stdout
)

logger = logging.getLogger(__name__)

In [ ]:
from numpy._core.defchararray import decode
# Neuronales Netz mit Autoencoder aufbauen
def build_and_train_autoencoder(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray
) -> Tuple[tf.keras.Model, float]:

  """
  Trainiert einen Autoencoder zur Anomalieerkennung von Fraud-Fällen.
  Trainiert ausschliesslich auf normalen Transaktionen.
  """

  # Daten filtern (nur normale Transaktionen)
  X_train_normal = X_train[y_train == 0]
  X_val_normal = X_val[y_val == 0]

  input_dim = X_train.shape[1]

  # Architektur des NN definieren (Encoder -> Bottleneck -> Decoder)
  input_layer = layers.Input(shape = (input_dim,))

  # Encoder (Komprimierung)
  encoded = layers.Dense(16, activation='relu')(input_layer)
  encoded = layers.Dropout(0.2)(encoded)                        # Overfitting verhindern
  encoded = layers.Dense(8, activation='relu')(encoded)        # Bottleneck

  # Decoder (Rekonstruktion)
  decoded = layers.Dense(16, activation='relu')(encoded)
  decoded = layers.Dense(input_dim, activation='linear')(decoded)

  # Modell zusammenbauen
  autoencoder = models.Model(inputs = input_layer, outputs = decoded)

  # Kompilieren
  autoencoder.compile(
      optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
      loss = 'mse'
  )

  # Early Stopping
  early_stopping = tf.keras.callbacks.EarlyStopping(
      monitor = 'val_loss',
      patience = 5,
      restore_best_weights = True
  )


  # Training
  autoencoder.fit(
      X_train_normal,
      X_train_normal,
      epochs = 50,
      batch_size = 256,
      validation_data = (X_val_normal, X_val_normal),
      callbacks = [early_stopping],
      verbose = 0
  )
  logger.info("Training des Autoencoders abgeschlossen.")

  # Evaluierung
  logger.info("Berechnung des Rekonstrukionsfehlers auf Validierungsset.")
  reconstructions = autoencoder.predict(X_val, verbose = 0)

  mse = np.mean(np.power(X_val - reconstructions, 2), axis = 1)
  aucpr = average_precision_score(y_val, mse)

  logger.info(f"Autoencoder AUCPR auf Validationset: {aucpr:.4f}")

  return autoencoder, aucpr

In [ ]:
autoencoder_model, ae_aucpr = build_and_train_autoencoder(X_train, y_train, X_val, y_val)

2026-03-30 17:23:30,440 - INFO - Training des Autoencoders abgeschlossen.
2026-03-30 17:23:30,442 - INFO - Berechnung des Rekonstrukionsfehlers auf Validierungsset.
2026-03-30 17:23:34,949 - INFO - Autoencoder AUCPR auf Validationset: 0.0574


XGBoost schliesst im Vergleich zum Autoencoder deutlich besser ab. Aus diesem Grund wird das XGBoost-Modell in einem nächsten Schritt weiter optimiert.

Optimierung des XGBoost-Modells

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report

In [ ]:
# Wahrscheinlichkeit auf dem Validierungsset vorhersagen
y_val_probs = best_model.predict_proba(X_val)[:, 1]

# Precision, Recall und Threshold berechnen
precision, recall, thresholds = precision_recall_curve(y_val, y_val_probs)

# Besten Threshold basierend auf dem F1-Score berechnen
f1_score = 2 * (precision * recall) / (precision + recall + 1e-12)
best_idx = np.argmax(f1_score)
optimal_threshold = thresholds[best_idx]

logger.info(f"Optimaler Threshold: {optimal_threshold:.4f}")
logger.info(f"Maximaler F1-Score auf Validation: {f1_score[best_idx]:.4f}")

2026-03-30 17:46:36,314 - INFO - Optimaler Threshold: 0.7349
2026-03-30 17:46:36,314 - INFO - Maximaler F1-Score auf Validation: 0.8235


In [ ]:
# Finale Evaluierung auf Testset
logger.info("Berechnung des AUCPR auf Testset.")

# Wahrscheinlichkeit für die Testdaten
y_test_probs = best_model.predict_proba(X_test)[:, 1]

# Vorhersage mit optimalem Threshold
y_test_pred = (y_test_probs >= optimal_threshold).astype(int)

# Ergebnis
print("\n" + "-" * 50)
print("Finales XGBoost-Modell auf Testset:")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_test_pred)
print(f"Echte Normale Transaktionen (TN) korrekt erkannt: {cm[0][0]}")
print(f"Falscher Alarm (FP):                              {cm[0][1]}")
print(f"Betrug übersehen (FN):                            {cm[1][0]}")
print(f"Betrug erfolgreich erkannt (TP):                  {cm[1][1]}")

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=["Normal", "Fraud"]))


2026-03-30 17:46:41,382 - INFO - Berechnung des AUCPR auf Testset.

--------------------------------------------------
Finales XGBoost-Modell auf Testset:

Confusion Matrix:
Echte Normale Transaktionen (TN) korrekt erkannt: 56880
Falscher Alarm (FP):                              7
Betrug übersehen (FN):                            20
Betrug erfolgreich erkannt (TP):                  55

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56887
       Fraud       0.89      0.73      0.80        75

    accuracy                           1.00     56962
   macro avg       0.94      0.87      0.90     56962
weighted avg       1.00      1.00      1.00     56962

